#### Goal here is to show the differences in the amt of proteins that have various properties in background vs significant...


In [ ]:
#load data...
#!
import pandas as pd 
import numpy as np

Full_DF = pd.read_csv("VTs_withprops_SLiMs.csv", header=0)

#There are two discrete ways to do this...

#1: Look at the enrichment by assigning based on the percentages of these 
# properties a binary classification. For example, all peptides with 0 - 50 "percent exposed"
# would be placed into the 'not-exposed' category, ect.
# then an enrichment test (fischer's, ect) could be performed on these two...

#2: Means difference and or variance difference statistical tests...
# Consider the distributions of a property, again 'percent exposed', for example
# then we can consider the distirbutions in the background and of the hits. 
# then we can end up seeing if the shape and the means of these distributions differ..
# like a KS test and or T test, dunno...



### 4/14/2026

Going to go for method #2 first and see what sort of statistical approaches would be appropriate for these distributions

In [ ]:
print(Full_DF.head())

print(f"Length: {len(Full_DF)}")



#want to make sure I have the right abt-unique anad rita-unique

print(f"Quantity of peptides with RITA data: {len(Full_DF.dropna(subset=['FC_RITA']))}")

In [ ]:
#4/17/2026. Later, want to make sure numbers are correct. Will be also mapping the Gene Names and the amino acid sequences

# Load positive-selection screen data.

from matplotlib import font_manager

from matplotlib import rcParams
rcParams['font.sans-serif'] = "Arial"
rcParams['font.family'] = "Arial"

df_rita_pos_selection = pd.read_excel(r"RITA_and_ABT_pos_selection_screens.xlsx",sheet_name = "RITA")
df_ABT_pos_selection = pd.read_excel(r"RITA_and_ABT_pos_selection_screens.xlsx",sheet_name = "ABT")

#merge these to get a mapping of the id to the amino acids, gene name

pos_selection_merged = pd.merge(df_rita_pos_selection, df_ABT_pos_selection, on="tileID", how='outer')

print(pos_selection_merged.head())

print(f"Length total: {len(pos_selection_merged)}")

print("Quantity of rita unique tiles prior to filtering", len(pos_selection_merged.dropna(subset=['Aminoacids_x'])['Aminoacids_x']))
print("Quantity of abt unique tiles prior to filtering", len(pos_selection_merged.dropna(subset=['Aminoacids_y'])['Aminoacids_y']))

#do these equal?

print(f"Are these equivalent? {pos_selection_merged.dropna(subset=['Aminoacids_x'])['Aminoacids_x'].equals(pos_selection_merged.dropna(subset=['Aminoacids_y'])['Aminoacids_y'])}")


#tileID col in these dataframes is equivalent to identifier in the above properties dataframe. I will map on this 

#merge on the ID with the RITA data



In [ ]:
#given that they are equal, we can now take the gene name and amino acids columns and append them to the properties WHERE applicable

In [ ]:
#lets go for Disorder, Helic, Coil_perc, Buried_perc, Exposed_perc. 


# Import plotting dependencies.
import seaborn as sns
import matplotlib.pyplot as plt 


#Drop all rows where there's NaN in the log2FoldChange

Full_DF = Full_DF.dropna(subset=['log2FoldChange'])


#get little subplots for some significant ones of interest...

RITA_Significant = Full_DF[Full_DF['Sig_RITA']=='Yes']



properties_string = 'Disorder_perc,Helix_perc,Coil_perc,Buried_perc,Exposed_perc'

properties_list = [x for x in properties_string.split(',')]



    

Ahh wait. Look at the plots above. They're negative... most of them, on their y-axes. That's not quite right, how can you have negative coiling? Let me check for 100% sure!!!



In [ ]:
#get minimum:

for property in properties_list:
    print(f"minimum for property {property}:", min(Full_DF[property]))
    print(f"maximum for property {property}:", max(Full_DF[property]))
    
#weird, weird. Let me just fix it with this next run:

In [ ]:

#To make things pretty later...
custom_palette=sns.color_palette(["#A5A8A8", "#36B1AD","#ECCC72"])

    


In [ ]:
#now we may break it down by the direction of that significance...

custom_palette=sns.color_palette(["#A5A8A8", "#36B1AD","#ECCC72"])
#we need a new col to express this....
def test_effect(row):
    
    if row['Sig_RITA'] == 'No':
        return 'None' 
    elif row['FC_RITA'] > 0:
        return 'Protective'
    else:
        return 'Deleterious'
    
Full_DF['Effect_RITA'] = Full_DF.apply(lambda row: test_effect(row), axis=1)

#run KS Tests...

from scipy import stats 
import numpy as np 
from scipy.stats import mannwhitneyu

significant_difference_dict_KS = {}
significant_difference_dict_MWU = {}
significant_difference_dict_UPT = {}

#compare background with other, for each...
for property in properties_list:
    background_dist = Full_DF[Full_DF['Effect_RITA']=='None'][property]
    Protective_RITA_dist = Full_DF[Full_DF['Effect_RITA'] == 'Protective'][property]
    Deleterious_RITA_dist = Full_DF[Full_DF['Effect_RITA'] == 'Deleterious'][property]
    
    
    print(f"Amount of peptides with no effect: {len(background_dist)}")
    print(f"Amount of peptides that were protective: {len(Protective_RITA_dist)}")
    print(f"Amount of peptides that were deleterious: {len(Deleterious_RITA_dist)}")
    
    #1: Compare background with protective dists
    
    protective_comparison = stats.kstest(background_dist, Protective_RITA_dist)
    deleterious_comparison = stats.kstest(background_dist, Deleterious_RITA_dist)
    
    #now we can obtain P-values:
    pvalue_protective_KS = protective_comparison[1]
    pvalue_deleterious_KS = deleterious_comparison[1]
    #now we can append this to the dict..
    
    significant_difference_dict_KS[property] = f"Protective: {round(pvalue_protective_KS,2)}\nDeleterious: {round(pvalue_deleterious_KS,2)}"
    
    U, pvalue_protective_MWU = stats.mannwhitneyu(background_dist, Protective_RITA_dist)
    U, pvalue_deleterious_MWU = stats.mannwhitneyu(background_dist, Deleterious_RITA_dist)
    
    significant_difference_dict_MWU[property] = f"Protective: {round(pvalue_protective_MWU,2)}\nDeleterious: {round(pvalue_deleterious_MWU,2)}"
    
    pvalue_protective_UPT = stats.ttest_ind(background_dist, Protective_RITA_dist, equal_var=False)[1]
    pvalue_deleterious_UPT = stats.ttest_ind(background_dist, Deleterious_RITA_dist, equal_var=False)[1]

    significant_difference_dict_UPT[property] = f"Protective: {round(pvalue_protective_UPT,2)}\nDeleterious: {round(pvalue_deleterious_UPT,2)}"
    
    
plot_stats = False
for property in properties_list:
    figure = plt.Figure(figsize=(20,16))
    ax = sns.violinplot(data=Full_DF, y=property, hue='Effect_RITA',palette=custom_palette, gap = 0.1, fill=True, inner='quart', cut=0)
    sns.despine(offset=10, trim=True)
    plt.ylim(0,100)
    plt.xlabel("")
    plt.legend(title='Peptide Effect')

    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    #we don't want perc for percocet in the y label for a nice looking figure lmao 
    current_ylabel  = ax.get_ylabel()
    new_ylabel = "Percent "+current_ylabel.split("_")[0]
    
    plt.ylabel(new_ylabel)
    plt.xticks([])
    
    if plot_stats:
        ax.text(0.6,0.5,f"KS-Test P-val Comparing none to\n{significant_difference_dict_KS[property]}", dict(size=(10)),bbox=dict(boxstyle='round', ec=(1.,0.5,0.5),fc=(1.,0.8,0.8)))
        ax.text(0.6,18,f"MWU-Test P-val Comparing none to\n{significant_difference_dict_MWU[property]}", dict(size=(10)), bbox=dict(boxstyle='round', ec=(1.,0.5,0.5),fc=(1.,0.8,0.8)))
        ax.text(0.6,35.5,f"Welch's T-test P-val Comparing none to\n{significant_difference_dict_UPT[property]}", dict(size=(10)), bbox=dict(boxstyle='round', ec=(1.,0.5,0.5),fc=(1.,0.8,0.8)))
        ax.text(0.6, 54.5, f"(with poly)\n# Protective: {len(Protective_RITA_dist)}\n# Deleterious: {len(Deleterious_RITA_dist)}", dict(size=(10)), bbox=dict(boxstyle='round', ec=(1.,0.5,0.5),fc=(1.,0.8,0.8)))
    plt.tight_layout()
    
    if plot_stats:
        plt.savefig(f"Property_{current_ylabel}_withstats.pdf",dpi=300)
        
    else:
        plt.savefig(f"Property_{current_ylabel}_nostats.pdf",dpi=300)

    plt.show()
    
    


In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplot_mosaic([['a','a','b','c'],['ca','ca','cb','cc']],
                              gridspec_kw={'wspace': 0, 'hspace': 0}, height_ratios=(15, 1), figsize=(6,4))
print(axs)

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

from matplotlib import font_manager

from matplotlib import rcParams
rcParams['font.sans-serif'] = "Arial"
rcParams['font.family'] = "Arial"

# Load the font directly from the download path



color_map = {'Exposed_perc': '#EF345C', 'Buried_perc': 'gray'}


#fig, axs = plt.subplots(1,3,figsize=(30,10),sharey=True)
sorted_full_df = Full_DF.sort_values(by=['Exposed_perc'])
sorted_full_df[['Exposed_perc', 'Buried_perc']].plot.bar(stacked=True, ax=axs['a'],width=2.00, color=color_map, edgecolor=None, alpha=0.0)

#sns.barplot(data=sorted_full_df, x=sorted_full_df.index, y='Exposed_perc', ax=axs['a'],width=1.00, color='black', fill=False)

prot_df_full = Full_DF[Full_DF['Effect_RITA']=='Protective']
sorted_prot_full = prot_df_full.sort_values(by=['Exposed_perc'])
sorted_prot_full[['Exposed_perc', 'Buried_perc']].plot.bar(stacked=True, ax=axs['b'],width=2.0, color=color_map, edgecolor=None, alpha=0.0)

# Calculate the continuous top line by summing the columns
top_line = sorted_prot_full['Exposed_perc']


del_df_full = Full_DF[Full_DF['Effect_RITA']=='Deleterious']
sorted_del_full = del_df_full.sort_values(by=['Exposed_perc'])
sorted_del_full[['Exposed_perc', 'Buried_perc']].plot.bar(stacked=True, ax=axs['c'],width=2.0, color=color_map, edgecolor=None, alpha=0.0)

#deleterious_sorted = Deleterious_RITA_dist.sort_values(by=['Exposed_perc'])
#deleterious_sorted.sort_values(by=['Exposed_perc']).plot.bar(stacked=True, ax=axs[2])

for pair in [(sorted_full_df, 'a'), (sorted_prot_full, 'b'), (sorted_del_full, ('c'))]:
    axs[pair[1]].plot(
    range(len(pair[0]['Exposed_perc'])),
    pair[0]['Exposed_perc'],
    color='black',
      linewidth=0.5,
      label='_nolegend_')
    axs[pair[1]].fill_between(range(len(pair[0]['Exposed_perc'])), pair[0]['Exposed_perc'], [0*len(pair[0]['Exposed_perc'])], fc='#EF345C', alpha=0.8)
    axs[pair[1]].fill_between(range(len(pair[0]['Exposed_perc'])),  [100*len(pair[0]['Exposed_perc'])],pair[0]['Exposed_perc'], fc='gray', alpha=0.8)


axs['a'].set_xticklabels([])
axs['b'].set_xticklabels([])
axs['c'].set_xticklabels([])



#axs['a'].set_title("Background")
#axs['b'].set_title("Protective")
#axs['c'].set_title("Deleterious")

axs['a'].set_xticks([])
axs['b'].set_xticks([])
axs['c'].set_xticks([])

axs['b'].set_yticklabels([])
axs['c'].set_yticklabels([])



for axis in ['ca', 'cb', 'cc']:
    axs[axis].set_xlim(0,1)
    axs[axis].set_ylim(0, -10)
    axs[axis].set_axis_off()

for axis in ['a','b','c']:
    axs[axis].set_ylim(0,100) 

axs['a'].legend([],[],frameon=False)
axs['b'].legend([],[],frameon=False)
axs['c'].legend(loc='center left', bbox_to_anchor=(1,0.5),edgecolor='white',labels=['Exposed', "Buried"])

axs['b'].set_yticks([])
axs['c'].set_yticks([])


axs['a'].set_ylabel("Proportion Predicted Exposed vs Buried")    
    
axs['ca'].barh(y=0, height=10, left=0, width=10, align='center', color="#A5A8A8")

axs['cb'].barh(y=0, height=10, left=0, width=10, align='center',color="#36B1AD")


axs['cc'].barh(y=0, height=10, left=0, width=10, align='center',color="#ECCC72")

axs['ca'].text(x=0.4, y=8, s="No Effect")

axs['cb'].text(x=0.22, y=8, s="Protective")

axs['cc'].text(x=0.21, y=8, s="Deleterious")


#plt.subplots_adjust(wspace=0, hspace=0)
custom_palette=sns.color_palette(["#A5A8A8", "#36B1AD","#ECCC72"])
plt.tight_layout()
plt.savefig("Exposed_Versus_Buried_Stackplot.pdf", dpi=300)

In [ ]:
from matplotlib.transforms import ScaledTranslation

fig, axs = plt.subplot_mosaic([['a','a','b','c'],['ca','ca','cb','cc']],
                              gridspec_kw={'wspace': 0, 'hspace': 0}, height_ratios=(15, 1), figsize=(6,4))
print(axs)

color_map = {'Disorder_perc': "#35CC83", 'Ordered': 'gray'}

Full_DF['Ordered'] = Full_DF['Disorder_perc'].apply(lambda x: 100-x)
#fig, axs = plt.subplots(1,3,figsize=(30,10),sharey=True)
sorted_full_df = Full_DF.sort_values(by=['Disorder_perc'])
sorted_full_df[['Disorder_perc', 'Ordered']].plot.bar(stacked=True, ax=axs['a'],width=1.00, color=color_map, alpha=0.0, edgecolor=None,)
prot_df_full = Full_DF[Full_DF['Effect_RITA']=='Protective']
sorted_prot_full = prot_df_full.sort_values(by=['Disorder_perc'])
sorted_prot_full[['Disorder_perc', 'Ordered']].plot.bar(stacked=True, ax=axs['b'],width=1.0, color=color_map, alpha=0.0, edgecolor=None)
del_df_full = Full_DF[Full_DF['Effect_RITA']=='Deleterious']
sorted_del_full = del_df_full.sort_values(by=['Disorder_perc'])
sorted_del_full[['Disorder_perc', 'Ordered']].plot.bar(stacked=True, ax=axs['c'],width=1.0, color=color_map, alpha=0.0, edgecolor=None)

#deleterious_sorted = Deleterious_RITA_dist.sort_values(by=['Exposed_perc'])
#deleterious_sorted.sort_values(by=['Exposed_perc']).plot.bar(stacked=True, ax=axs[2])

for pair in [(sorted_full_df, 'a'), (sorted_prot_full, 'b'), (sorted_del_full, ('c'))]:
    axs[pair[1]].plot(
    range(len(pair[0]['Disorder_perc'])),
    pair[0]['Disorder_perc'],
    color='black',
      linewidth=0.5,
      label='_nolegend_'

)
    axs[pair[1]].fill_between(range(len(pair[0]['Disorder_perc'])), pair[0]['Disorder_perc'], [0*len(pair[0]['Disorder_perc'])], fc="#35CC83", alpha=0.8)
    axs[pair[1]].fill_between(range(len(pair[0]['Disorder_perc'])), [100*len(pair[0]['Disorder_perc'])],pair[0]['Disorder_perc'], fc="gray")


axs['a'].set_xticklabels([])
axs['b'].set_xticklabels([])
axs['c'].set_xticklabels([])



#axs['a'].set_title("Background")
#axs['b'].set_title("Protective")
#axs['c'].set_title("Deleterious")

axs['a'].set_xticks([])
axs['b'].set_xticks([])
axs['c'].set_xticks([])

axs['b'].set_yticklabels([])
axs['c'].set_yticklabels([])


for axis in ['ca', 'cb', 'cc']:
    axs[axis].set_xlim(0,1)
    axs[axis].set_ylim(0, -10)
    axs[axis].set_axis_off()

for axis in ['a','b','c']:
    axs[axis].set_ylim(0,100) 

axs['a'].legend([],[],frameon=False)
axs['b'].legend([],[],frameon=False)
axs['c'].legend(loc='center left', bbox_to_anchor=(1,0.5),edgecolor='white',labels=['Disordered', "Ordered"])


axs['a'].set_ylabel("Percent Predicted Ordered vs Disordered")    
    
axs['ca'].barh(y=0, height=10, left=0, width=10, align='center', color="#A5A8A8")

axs['cb'].barh(y=0, height=10, left=0, width=10, align='center',color="#36B1AD")

axs['cc'].barh(y=0, height=10, left=0, width=10, align='center',color="#ECCC72")

axs['b'].set_yticks([])
axs['c'].set_yticks([])
# Hide the left Y-axis spine


axs['ca'].text(x=0.4, y=8, s="No Effect")

axs['cb'].text(x=0.22, y=8, s="Protective")

axs['cc'].text(x=0.20, y=8, s="Deleterious")

#plt.subplots_adjust(wspace=0, hspace=0)
custom_palette=sns.color_palette(["#A5A8A8", "#36B1AD","#ECCC72"])
plt.tight_layout()
plt.savefig("Disorder_Versus_Order_Stackplot.pdf", dpi=300)